# 라벨 ID에서 사용자명 추출

원본 최종 수정 기록: 2025-05-31T13:52:22+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.

팀 기여: 계정 데이터 크롤링은 팀원 담당 영역입니다. 본인은 수집 이후 전처리·정규화·모델 학습과 PM 역할을 담당했습니다.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import time
# CSV 파일 불러오기
df = pd.read_csv("sampled_label.csv")  # 기존 데이터 파일 (id, label)
df["id"] = df["id"].astype(str).str.replace("^u", "", regex=True)  # "u" 제거

In [ ]:
df.head()

In [ ]:

# 🔹 Selenium 설정 (Chrome Headless 모드)
options = Options()
options.add_argument("--headless")  # 브라우저 창 없이 실행
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")

test_id = 17140361
# 🔹 Chrome 드라이버 실행
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

user_ids = df["id"].tolist()

# 🔹 결과 저장
id_to_username = {}

for user_id in test_id:
    try:
        url = f"https://twitter.com/i/user/{user_id}"
        driver.get(url)
        time.sleep(3)  # 페이지 로딩 대기

        # 🔹 리디렉션된 URL에서 @username 추출
        redirected_url = driver.current_url  # 최종 URL (예: https://twitter.com/elonmusk)
        username = redirected_url.split("/")[-1]  # 마지막 부분이 @username

        id_to_username[user_id] = username
        print(f"✅ {user_id} -> @{username}")

    except Exception as e:
        print(f"❌ {user_id} 크롤링 실패: {e}")
        id_to_username[user_id] = None

# 🔹 Selenium 종료
driver.quit()

# 🔹 결과를 CSV 파일로 저장
df["username"] = df["id"].map(id_to_username)  # ID 매핑
df.to_csv("twitter_usernames.csv", index=False)

print("🚀 크롤링 완료! 'twitter_usernames.csv' 파일 저장됨.")

In [ ]:
pip install -U pybind11

In [ ]:
pip install -U scikit-learn